# 06 · Outcome models and feasible policy validation
Squared-error NCM prediction is a diagnostic; primary decision evidence is held-out constrained policy value.
Original XGBoost configuration is tuned on CP06. Candidate policies use nested rolling development folds CP04–06; the choice is locked before retrospective CP07 evaluation.

## Load data

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from src.common import TABLES, FIGS, MODELS, load_processed
from src.modeling import ACTIONS, COMMERCIAL, XGB_FIXED, XGB_GRID, feature_lists, make_xgb_pipe, reg_metrics
pd.set_option("display.width", 200)
df = load_processed("historical_feature_outcome")
NUM, CAT = feature_lists()
X_COLS = NUM + CAT
outcomes = set(pd.read_csv(TABLES / "outcome_schema.csv").outcome)
FORBIDDEN = {"assignment_id", "customer_id", "campaign_id", "send_ts", "assignment_source", "offer_type", "email_delivered"} | outcomes
assert len(X_COLS) == 43 and not set(X_COLS) & FORBIDDEN
exp = df[df.assignment_source == "exploration"].copy()
SPLIT = {"train": [f"CP0{i}" for i in range(1, 6)], "valid": ["CP06"], "test": ["CP07"]}
print("exploration rows:", len(exp), "| predictors:", len(X_COLS))
pd.crosstab(exp.campaign_id, exp.offer_type, margins=True)

exploration rows: 7527 | predictors: 43


offer_type,FREE_SHIP,NO_OFFER,PCT_10,PCT_20,All
campaign_id,,,,,
CP01,265,259,271,269,1064
CP02,291,254,262,303,1110
CP03,269,278,256,253,1056
CP04,283,257,288,289,1117
CP05,249,260,246,270,1025
CP06,283,261,285,266,1095
CP07,266,257,294,243,1060
All,1906,1826,1902,1893,7527


## Helpers — fit four action models / factual and counterfactual scoring

In [2]:
def fit_tlearner(d, params):
    return {a: make_xgb_pipe(NUM, CAT, params).fit(d.loc[d.offer_type == a, X_COLS], d.loc[d.offer_type == a, "ncm_14d"]) for a in ACTIONS}

def factual(models, d):
    p = pd.Series(np.nan, index=d.index)
    for a, m in models.items():
        idx = d.offer_type == a
        p[idx] = m.predict(d.loc[idx, X_COLS])
    return p

def counterfactual(models, d):
    out = pd.DataFrame({f"predicted_ncm_{a}": models[a].predict(d[X_COLS]) for a in ACTIONS}, index=d.index)
    for a in ACTIONS:
        out[f"incremental_ncm_{a}"] = out[f"predicted_ncm_{a}"] - out["predicted_ncm_NO_OFFER"]
    out["recommended_action_unconstrained"] = out[[f"predicted_ncm_{a}" for a in ACTIONS]].idxmax(axis=1).str.replace("predicted_ncm_", "")
    return out

tr, va, te = (exp[exp.campaign_id.isin(SPLIT[k])] for k in ("train", "valid", "test"))

## Tune one shared configuration on CP06 (pooled factual RMSE, MAE tie-break)

In [3]:
grid = []
for params in XGB_GRID:
    grid.append({**params, **reg_metrics(va.ncm_14d, factual(fit_tlearner(tr, params), va))})
mean_base = va.offer_type.map(tr.groupby("offer_type").ncm_14d.mean())
grid.append({"n_estimators": "action-mean baseline", **reg_metrics(va.ncm_14d, mean_base)})
grid = pd.DataFrame(grid)
grid.to_csv(TABLES / "tlearner_validation_grid.csv", index=False)
best = grid[grid.n_estimators != "action-mean baseline"].sort_values(["rmse", "mae"]).iloc[0]
BEST = {"n_estimators": int(best.n_estimators), "max_depth": int(best.max_depth), "learning_rate": float(best.learning_rate)}
print("selected:", BEST)
grid[["n_estimators", "max_depth", "learning_rate", "rmse", "mae", "r2"]].round(4)

selected: {'n_estimators': 100, 'max_depth': 1, 'learning_rate': 0.03}


,n_estimators,max_depth,learning_rate,rmse,mae,r2
0,200,2.0,0.03,65.8633,29.6648,0.0011
1,200,2.0,0.05,66.1365,29.7075,-0.0073
2,200,3.0,0.03,66.2773,29.5629,-0.0115
3,200,3.0,0.05,66.7458,29.9680,-0.0259
4,400,2.0,0.03,66.2723,29.9354,-0.0114
5,400,2.0,0.05,67.0164,30.4962,-0.0342
6,400,3.0,0.03,66.9463,30.1154,-0.0321
7,400,3.0,0.05,67.9456,31.4262,-0.0631
8,100,1.0,0.03,65.4966,28.8979,0.0121
9,200,1.0,0.03,65.5295,29.4945,0.0111


## Complete policy backtest and conservative model selection
Each fold trains on earlier campaigns and tunes on its latest prior campaign. All policies share the same holdout, cost predictions and constraints. DR/Hájek estimates use randomized observed outcomes, with customer-bootstrap paired intervals. No replacement is made unless the development evidence clears a multiplicity-adjusted improvement threshold. CP07 was inspected in the prior audit: this is a retrospective held-out analysis, not a fresh prospective test.

In [4]:
from src.evaluation import run_policy_validation
from src.modeling import predict_actions
selection, test_result = run_policy_validation(df, NUM, CAT, BEST, TABLES, MODELS)
selected = selection['selected_method']
selected_metrics = test_result['metrics'].query('model == @selected').drop(columns=['split', 'model'])
selected_metrics.to_csv(TABLES / 'tlearner_test_metrics.csv', index=False)
print('Selected model:', selected)
display(test_result['summary'][['policy','dr_incremental_ncm_per_customer','ci_low','ci_high','offered_share','matched_rows']])
display(test_result['metrics'].query("scope == 'pooled'"))


Policy development fold CP04


Policy development fold CP05


Policy development fold CP06


Selection locked: ORIGINAL_TLEARNER; now evaluating CP07


Selected model: ORIGINAL_TLEARNER


,policy,dr_incremental_ncm_per_customer,ci_low,ci_high,offered_share,matched_rows
0,ORIGINAL_TLEARNER,3.971406,-2.233164,9.682505,0.6,251
1,SHRUNK_TLEARNER,4.891893,-1.324456,11.015848,0.6,251
2,ACTION_MEAN,5.571300,-2.097807,13.623382,0.6,271
3,POOLED_XGBOOST,6.228061,-1.994401,15.601585,0.6,242
4,ALL_NO_OFFER,0.000000,0.000000,0.000000,0.0,257
5,PROPENSITY_PCT10,3.650978,-2.492067,9.759834,0.6,258
6,HISTORICAL_MIX,4.659854,-0.483924,9.758685,0.6,237
7,RANDOM_FREE_SHIP,12.741429,3.589919,22.663654,0.6,246


,split,model,scope,n,rmse,mae,r2,obs_mean,pred_mean
0,CP07,ORIGINAL_TLEARNER,pooled,1060,63.369408,27.411143,0.005863,15.31483,16.573692
5,CP07,SHRUNK_TLEARNER,pooled,1060,63.225679,27.272264,0.010368,15.31483,15.986402
10,CP07,ACTION_MEAN,pooled,1060,63.488908,27.161594,0.002110,15.31483,15.399111
15,CP07,POOLED_XGBOOST,pooled,1060,62.832872,27.232219,0.022626,15.31483,17.093073


## Save compatible scoring outputs and metadata
The selected action models are already refit on CP01–07 and saved by the validator. Factual regression scores, unconstrained scores, and in-sample effect averages are descriptive diagnostics; the constrained policy evidence is in heldout_policy_comparison.csv.

In [5]:
te = test_result['test']
mu = test_result['predictions'][selected]
pred = te[['assignment_id','customer_id','campaign_id','offer_type','ncm_14d']].copy()
for j,a in enumerate(ACTIONS):
    pred[f'predicted_ncm_{a}'] = mu[:,j]
    pred[f'incremental_ncm_{a}'] = mu[:,j]-mu[:,0]
pred['recommended_action_unconstrained'] = np.asarray(ACTIONS)[mu.argmax(axis=1)]
pred.to_csv(TABLES / 'tlearner_test_predictions.csv', index=False)
test_result['summary'].assign(policy_scope='feasible constrained policy').to_csv(TABLES / 'tlearner_policy_value_cp07.csv', index=False)
final = {a: joblib.load(MODELS / f'tlearner_ncm_{a}.joblib') for a in ACTIONS}
final_mu = predict_actions(final, exp[X_COLS])
step04 = pd.read_csv(TABLES / 'incremental_ncm_exploration.csv').query("segment == 'exploration' and outcome == 'ncm_14d'").set_index('offer_type')
effects = pd.DataFrame({'action':ACTIONS,'model_mean_incremental_ncm_CP07':(mu-mu[:,[0]]).mean(axis=0),
    'final_model_mean_incremental_ncm_all_exploration':(final_mu-final_mu[:,[0]]).mean(axis=0),
    'step04_randomized_effect_CP01_07':[0.] + [step04.loc[a,'incremental_ncm'] for a in COMMERCIAL],
    'interpretation':'final-model historical averages are in-sample sanity checks, not independent validation'})
effects.to_csv(TABLES / 'tlearner_action_effect_summary.csv',index=False)
meta={'target':'ncm_14d','training_population':'assignment_source == exploration','actions':ACTIONS,
      'numeric_features':NUM,'categorical_features':CAT,'xgb_fixed':XGB_FIXED,'xgb_selected':BEST,
      'selected_method':selected,'selection':selection['rule'],
      'final_train_rows':{a:int(exp.offer_type.eq(a).sum()) for a in ACTIONS},
      'cp07_pooled':selected_metrics.query("scope == 'pooled'").iloc[0][['rmse','mae','r2']].to_dict()}
(MODELS/'tlearner_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
# Gain is descriptive; it does not identify causes or validate treatment effects.
importance=[]
for a,m in final.items():
    model=getattr(m,'model',m)
    if not hasattr(model,'steps'): continue
    names=model[0].get_feature_names_out()
    for k,gain in model[-1].get_booster().get_score(importance_type='gain').items():
        feature=names[int(k[1:])] if k.startswith('f') and k[1:].isdigit() else k
        importance.append({'action':a,'encoded_feature':feature,'gain':gain})
pd.DataFrame(importance,columns=['action','encoded_feature','gain']).to_csv(TABLES/'tlearner_feature_importance.csv',index=False)
display(selected_metrics.round(4))


,scope,n,rmse,mae,r2,obs_mean,pred_mean
0,pooled,1060,63.3694,27.4111,0.0059,15.3148,16.5737
1,NO_OFFER,257,40.9644,23.4051,-0.0387,10.2449,17.8076
2,PCT_10,294,74.5545,27.1283,-0.0042,16.1856,14.7944
3,PCT_20,243,36.7600,18.5409,0.0418,11.3396,9.6422
4,FREE_SHIP,266,83.6846,39.6975,0.0041,22.8823,23.6802


## Diagnostics: predictive error versus feasible policy value
Intervals below refer to incremental NCM relative to no offers, conditional on fitted models and this campaign. They do not measure July transport uncertainty.

In [6]:
fig,ax=plt.subplots(1,2,figsize=(13,4.5))
metrics=test_result['metrics'].query("scope == 'pooled'")
ax[0].barh(metrics.model,metrics.rmse,color='#2a78d6'); ax[0].set_xlabel('CP07 factual RMSE ($)')
pv=test_result['summary'].set_index('policy')
y=np.arange(len(pv)); x=pv.dr_incremental_ncm_per_customer.to_numpy()
ax[1].errorbar(x,y,xerr=np.maximum(np.vstack([x-pv.ci_low.to_numpy(),pv.ci_high.to_numpy()-x]),0),fmt='o',color='#2a78d6')
ax[1].set_yticks(y,pv.index); ax[1].axvline(0,color='grey',ls='--'); ax[1].set_xlabel('CP07 feasible-policy incremental NCM ($/customer)')
fig.tight_layout(); fig.savefig(FIGS/'06_tlearner_diagnostics.png',dpi=120,bbox_inches='tight'); plt.show()


C:\Users\yifan\AppData\Local\Temp\ipykernel_45248\3231617418.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'06_tlearner_diagnostics.png',dpi=120,bbox_inches='tight'); plt.show()
